# 04 – Hvaða flokk myndi líkanið spá fyrir um þig?

Hér geturðu **slegið inn þín eigin svör** við spurningum úr ÍSKOS-könnuninni
og séð hvað hvert af þjálfuðu líkönunum myndi spá fyrir um — bæði sem
**tölfræðilegar líkur fyrir hvern flokk** og með **SHAP-túlkun** sem sýnir
hvaða svör höfðu mest áhrif á forspána.

**Leiðbeiningar:**
1. Keyrðu fyrstu tvær frumurnar (innflutningar + sjálfgildi).
2. Breyttu `my_responses`-orðabókinni að vild. Sjálfgildin eru
   miðgildi/algengasta gildi úr þjálfunarúrtakinu — sleppt þeim sem þú
   vilt ekki tjá þig um.
3. Keyrðu áfram til að sjá forspár og SHAP-mynd fyrir hvert líkan.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = PROJECT_ROOT / "data" / "processed"
DATA_PATH = PROJECT_ROOT / "Kjósendakönnun" / "ÍSKOS kjósenda 2024" / "ISKOS2024_med_aukabreytum_20260318.sav"
MODELS = PROJECT_ROOT / "models"

_SRC = str((PROJECT_ROOT / "src").resolve())
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)
from iskos_labels import IskosLabels  # noqa: E402
labels = IskosLabels.from_sav(DATA_PATH, feature_groups_path=PROC / "feature_groups.json")

# Load processed data and metadata
train_df = pd.read_parquet(PROC / "train.parquet")
test_df = pd.read_parquet(PROC / "test.parquet")
with (PROC / "feature_groups.json").open() as f:
    fg = json.load(f)
with (PROC / "target_labels.json").open() as f:
    tl = json.load(f)

BASE_FEATURES = fg["base_features"]
NOMINAL_COLS = set(fg["nominal_columns"])
NUMERIC_COLS = set(fg["numeric_columns"])

# Load every trained tree-based model we want to interrogate
MODELS_TO_TRY = {
    "Decision tree": joblib.load(MODELS / "decision_tree.pkl"),
    "Random forest (naive)": joblib.load(MODELS / "rf_base.pkl"),
    "Random forest (balanced)": joblib.load(MODELS / "rf_base_balanced.pkl"),
    "Gradient boosting (naive)": joblib.load(MODELS / "histgb_base.pkl"),
    "Gradient boosting (balanced)": joblib.load(MODELS / "histgb_base_balanced.pkl"),
}

print(f"Hlaðin {len(MODELS_TO_TRY)} líkön: " + ", ".join(MODELS_TO_TRY))
print(f"Forspárbreytur (grunn): {len(BASE_FEATURES)}")

## 1 — Sjálfgefin svör

Hér reiknum við **miðgildi** fyrir tölulegar breytur og **algengasta gildi**
fyrir flokkabreytur úr þjálfunarúrtakinu. Þetta verður útgangspunkturinn —
ef þú breytir engu er forspáin sú sem dæmigerður svarandi fær.

In [ ]:
# Build default values for every base feature
defaults = {}
for col in BASE_FEATURES:
    s = train_df[col]
    if col in NOMINAL_COLS:
        # Mode of the categorical column (ignore "Missing")
        non_missing = s[s != "Missing"]
        defaults[col] = str(non_missing.mode().iloc[0]) if len(non_missing) else "Missing"
    else:
        defaults[col] = float(s.median())

# Quick sanity print
print("Fyrstu 10 sjálfgildi:")
for k in list(defaults)[:10]:
    val_label = ""
    if k in NOMINAL_COLS:
        val_label = f"  ({labels.value_label(k, defaults[k])})"
    print(f"  {k:18s} = {defaults[k]!r:8}{val_label}")

## 2 — Tilvalin gildi fyrir hverja breytu

Hér prentum við út **hvaða gildi eru leyfileg** fyrir hverja flokkabreytu og
**hvaða svið** tölulegar breytur taka. Notaðu þetta sem leiðarvísi þegar þú
breytir `my_responses` í næstu frumu.

In [ ]:
def show_options(var: str) -> None:
    print(f"\n{var} — {labels.short_question(var)}")
    print(f"  Spurning: {labels.question(var)}")
    if var in NOMINAL_COLS:
        # All known value labels for this categorical
        opts = labels.value_labels.get(var, {})
        if opts:
            for code, lab in opts.items():
                print(f"    {code:>4s}  {lab}")
        else:
            # Fallback: just show what categories exist in training data
            for cat in train_df[var].cat.categories[:15]:
                print(f"    {cat}")
    else:
        s = train_df[var]
        print(f"  Töluleg breyta: lágm.={s.min()}, miðgildi={s.median()}, hám.={s.max()}")

# Show a couple of examples — the rest follow the same pattern
for v in ["lrscale", "educationflo", "elecdist", "govprfm", "EUmship"]:
    show_options(v)

## 3 — Þín svör (`my_responses`)

**Breyttu hér.** Hver lykill er ein spurning; gildin eru annaðhvort tölur
(fyrir tölulegar breytur) eða strengir (fyrir flokkabreytur). Athugasemdir
til hægri sýna hvaða spurning er undir og — fyrir flokkabreytur — hvaða
svar samsvarar sjálfgildinu.

Allt er forfyllt með „dæmigerðu" svari. Þú þarft einungis að breyta því sem
þú vilt prófa.

In [ ]:
my_responses = {
    # gender — see show_options("gender")
    "gender": defaults["gender"],
    # age — see show_options("age")
    "age": defaults["age"],
    # educationflo — see show_options("educationflo")
    "educationflo": defaults["educationflo"],
    # elecdist — see show_options("elecdist")
    "elecdist": defaults["elecdist"],
    # urban — see show_options("urban")
    "urban": defaults["urban"],
    # emplstatus — see show_options("emplstatus")
    "emplstatus": defaults["emplstatus"],
    # incomecate — see show_options("incomecate")
    "incomecate": defaults["incomecate"],
    # rentown — see show_options("rentown")
    "rentown": defaults["rentown"],
    # maritals — see show_options("maritals")
    "maritals": defaults["maritals"],
    # hhmmb — see show_options("hhmmb")
    "hhmmb": defaults["hhmmb"],
    # childs — see show_options("childs")
    "childs": defaults["childs"],
    # languagecat — see show_options("languagecat")
    "languagecat": defaults["languagecat"],
    # church — see show_options("church")
    "church": defaults["church"],
    # rlgatnd — see show_options("rlgatnd")
    "rlgatnd": defaults["rlgatnd"],
    # lrscale — see show_options("lrscale")
    "lrscale": defaults["lrscale"],
    # polintr — see show_options("polintr")
    "polintr": defaults["polintr"],
    # pltrust — see show_options("pltrust")
    "pltrust": defaults["pltrust"],
    # envirimp — see show_options("envirimp")
    "envirimp": defaults["envirimp"],
    # gincdif — see show_options("gincdif")
    "gincdif": defaults["gincdif"],
    # taxreduc — see show_options("taxreduc")
    "taxreduc": defaults["taxreduc"],
    # capicntry — see show_options("capicntry")
    "capicntry": defaults["capicntry"],
    # ceqvote — see show_options("ceqvote")
    "ceqvote": defaults["ceqvote"],
    # privhealth — see show_options("privhealth")
    "privhealth": defaults["privhealth"],
    # imueclt — see show_options("imueclt")
    "imueclt": defaults["imueclt"],
    # essfull — see show_options("essfull")
    "essfull": defaults["essfull"],
    # femfight — see show_options("femfight")
    "femfight": defaults["femfight"],
    # banks — see show_options("banks")
    "banks": defaults["banks"],
    # immrght — see show_options("immrght")
    "immrght": defaults["immrght"],
    # genconsind — see show_options("genconsind")
    "genconsind": defaults["genconsind"],
    # acrproimp — see show_options("acrproimp")
    "acrproimp": defaults["acrproimp"],
    # drugpol — see show_options("drugpol")
    "drugpol": defaults["drugpol"],
    # gndequ — see show_options("gndequ")
    "gndequ": defaults["gndequ"],
    # EUmship — see show_options("EUmship")
    "EUmship": defaults["EUmship"],
    # govprfm — see show_options("govprfm")
    "govprfm": defaults["govprfm"],
    # ecobettwors_1 — see show_options("ecobettwors_1")
    "ecobettwors_1": defaults["ecobettwors_1"],
}

# Show what the dict currently says, with a readable label per value
print("Þín svör:")
for col, v in my_responses.items():
    short = labels.short_question(col)
    if col in NOMINAL_COLS:
        readable = labels.value_label(col, v)
        print(f"  {short:32s} = {v!r:<8} ({readable})")
    else:
        print(f"  {short:32s} = {v}")

## 4 — Búum til forspárinntak fyrir líkönin

Líkönin búast við töflu með nákvæmlega sömu dálkum og þjálfunarúrtakið. Hér
útbúum við einnar-línu töflu (`my_input`) og setjum réttar gagnagerðir
(`Categorical` fyrir flokkabreytur) svo `OneHotEncoder` í forvinnsluleiðslu
hvers líkans virki rétt.

In [ ]:
def build_my_input(responses: dict) -> pd.DataFrame:
    """Convert the user's response dict into a single-row DataFrame
    that matches the schema (categories included) of train_df[BASE_FEATURES]."""
    row = {}
    for col in BASE_FEATURES:
        v = responses.get(col, defaults[col])
        if col in NOMINAL_COLS:
            # Cast to the SAME Categorical dtype as training data
            cats = train_df[col].cat.categories
            v_str = str(v) if v in cats else "Missing"
            row[col] = pd.Categorical([v_str], categories=cats)
        else:
            row[col] = [float(v)]
    df = pd.DataFrame(row)
    return df

my_input = build_my_input(my_responses)
my_input

## 5 — Forspár hvers líkans

Tafla yfir **líkur fyrir hvern flokk** samkvæmt hverju líkani. Líkanið með
hæstu líkur fyrir tiltekinn flokk er valið sem forspá þess.

In [ ]:
def predict_one(model_pipeline, X_single: pd.DataFrame) -> pd.Series:
    """Return per-class probabilities as a pandas Series with class labels as index."""
    proba = model_pipeline.predict_proba(X_single)[0]
    return pd.Series(proba, index=model_pipeline.classes_)

# Build a wide DataFrame: rows = classes, columns = models
proba_table = pd.DataFrame({
    name: predict_one(m, my_input) for name, m in MODELS_TO_TRY.items()
})
# Render with party labels (Icelandic)
proba_display = proba_table.copy()
proba_display.index = [tl["target_class_labels_is"].get(c, c) for c in proba_display.index]
# Highlight the argmax per column with a marker for human reading
print("Líkindi fyrir hvern flokk (%):")
for col in proba_display.columns:
    pred = proba_display[col].idxmax()
    print(f"\n  {col}  →  forspá: {pred} ({proba_display[col].max() * 100:.1f}%)")
proba_display.style.format("{:.1%}").background_gradient(cmap="Blues", axis=0)

In [ ]:
# Visual summary: bar chart per model showing prob for each class (top 4 in each)
fig, axes = plt.subplots(1, len(MODELS_TO_TRY), figsize=(4.6 * len(MODELS_TO_TRY), 5), sharey=True)
if len(MODELS_TO_TRY) == 1:
    axes = [axes]
for ax, (name, _) in zip(axes, MODELS_TO_TRY.items()):
    series = proba_display[name].sort_values(ascending=True)
    colors = ["#c75b5b" if i == len(series) - 1 else "#4a90a4" for i in range(len(series))]
    ax.barh(series.index, series.values * 100, color=colors)
    ax.set_xlim(0, 100)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Líkur (%)")
plt.suptitle("Forspáð líkindi fyrir hvern flokk – einn svarandi", y=1.02)
plt.tight_layout()
plt.show()

## 6 — Hvaða svör höfðu mest áhrif? (SHAP-skýring fyrir hvert líkan)

Fyrir hvert líkan reiknum við **SHAP-gildi fyrir nákvæmlega þetta eina svör**
og sýnum *vatnsfallsmynd* (*waterfall plot*) sem segir hvaða svör þrýstu
forspánni upp og niður — fyrir þann flokk sem líkanið kaus.

Öll trjáa-líkön nota `shap.TreeExplainer`, sem reiknar SHAP-gildi nákvæmlega
fyrir trjáa-byggð líkön.

In [ ]:
def shap_waterfall_for_single(name: str, model_pipeline, X_single: pd.DataFrame) -> None:
    """Render a SHAP waterfall plot for the model's predicted class on this single observation."""
    prep = model_pipeline.named_steps["prep"]
    clf = model_pipeline.named_steps["clf"]

    # Transform input through the SAME preprocessor used during training
    X_t = prep.transform(X_single)
    feat_names = [labels.readable(n) for n in prep.get_feature_names_out()]

    # Predicted class for this observation
    proba = clf.predict_proba(X_t)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = clf.classes_[pred_idx]
    pred_label_is = tl["target_class_labels_is"].get(pred_label, pred_label)

    # Compute SHAP. For tree models on multiclass problems, shap_values can be
    # either a list (one array per class) or a 3D array (n_samples, n_features, n_classes).
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_t)
    if isinstance(sv, list):
        shap_for_class = sv[pred_idx][0]
        base_value = explainer.expected_value[pred_idx]
    elif isinstance(sv, np.ndarray) and sv.ndim == 3:
        shap_for_class = sv[0, :, pred_idx]
        base_value = (
            explainer.expected_value[pred_idx]
            if isinstance(explainer.expected_value, (list, np.ndarray))
            else explainer.expected_value
        )
    else:
        # Binary or 2D fallback
        shap_for_class = sv[0]
        base_value = explainer.expected_value

    # Build a shap.Explanation and render
    expl = shap.Explanation(
        values=shap_for_class,
        base_values=base_value,
        data=X_t[0],
        feature_names=feat_names,
    )
    print(f"\n{name} — forspáð flokkur: {pred_label_is} (líkur {proba[pred_idx] * 100:.1f}%)")
    shap.plots.waterfall(expl, max_display=12, show=False)
    plt.title(f"{name} — SHAP fyrir forspána '{pred_label_is}'", fontsize=11)
    plt.tight_layout()
    plt.show()

# One waterfall per model
for name, model in MODELS_TO_TRY.items():
    shap_waterfall_for_single(name, model, my_input)

## Samantekt

Þú hefur núna séð:

1. **Líkindaforspá hvers líkans** fyrir þín svör.
2. **SHAP-vatnsfallsmynd** sem útskýrir hvaða svör þrýstu forspánni í þá
   átt sem líkanið kaus.
3. **Mismunandi líkön gefa stundum mismunandi forspár** — þetta er fast
   einkenni vélnáms: enginn einn „réttur" reiknirit, og ákvarðanir um
   leiðréttingu fyrir ójafnvægi (`balanced` vs `naive`) skipta máli.

Prófaðu að breyta nokkrum gildum í `my_responses` — t.d. færa `lrscale` eins
langt til vinstri eða hægri og hægt er, eða breyta menntun og kjördæmi — og
sjáðu hvernig forspáin breytist.